# 03 — MutationEngine: Evolutionary Strategy Search

The MutationEngine searches for high-performing (ticker, strategy, params) combinations using a simple evolutionary algorithm:

1. **Seed** — random population of (ticker, strategy, params) individuals
2. **Evaluate** — backtest each, score by Sharpe ratio
3. **Select** — keep the top performers
4. **Mutate** — perturb params, swap tickers, add fresh randoms
5. **Repeat** for N generations

> ⚠️ **Data snooping warning:** Results from this engine are in-sample by construction.
> You are running hundreds of backtests and keeping the best-looking ones — this *will*
> inflate Sharpe ratios. Any strategy that looks interesting here must be validated
> on out-of-sample data before you trust it.

---

In [1]:
import sys
sys.path.insert(0, '../../')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt

from DrakonixBacktester.mutationengine import MutationEngine
from DrakonixBacktester.mutationengine.gui import ResultsBrowser

plt.rcParams['figure.figsize'] = (13, 6)

---

## 1. Configure and Run

The engine searches over 12 tickers × 5 strategy types × random parameter ranges.

**Tickers:** SPY, QQQ, IWM, AAPL, MSFT, NVDA, GOOGL, AMZN, META, JPM, XOM, JNJ  
**Strategies:** SMACrossover, BollingerBands, DonchianBreakout, TimeSeriesMomentum, RSITrendFilter  
**Fitness:** Sharpe ratio (min 5 trades required)

Adjust `population_size` and `generations` to trade runtime for thoroughness.

In [2]:
engine = MutationEngine(
    start='2015-01-01',
    end='2025-01-01',
    initial_capital=10_000,
    commission=0.001,
    min_trades=5,
    fitness='sharpe',
    seed=42,
)

results = engine.run(
    generations=3,
    population_size=60,
    top_k=10,
)

  Loaded 12 tickers successfully.
Generation 1/3 — evaluating 60 individuals...
  53 valid | top sharpe: 1.375 (NVDA | TimeSeriesMomentum(lookback=72, skip=7))
Generation 2/3 — evaluating 60 individuals...
  58 valid | top sharpe: 1.505 (NVDA | TimeSeriesMomentum(lookback=109, skip=3))
Generation 3/3 — evaluating 60 individuals...
  59 valid | top sharpe: 1.578 (NVDA | TimeSeriesMomentum(lookback=152, skip=7))


---

## 2. Top Results Table

In [ ]:
# Display top 20, hiding the _result column
display_cols = ['ticker', 'strategy', 'params', 'sharpe', 'total_return', 'max_drawdown', 'cagr', 'n_trades', 'generation']
print('Top 20 strategies by Sharpe ratio:\n')
print(results[display_cols].head(20).to_string(index=True))

---

## 3. Overlay: Top 10 Equity Curves

In [ ]:
browser = ResultsBrowser(results, initial_capital=10_000, top_n=20)
browser.plot_all(top_n=10)

---

## 4. Interactive Side-by-Side Browser

Use the dropdowns to select any two strategies from the top results and compare their equity curves, drawdowns, and stats side by side. The chart updates automatically when you change selections.

In [ ]:
browser.show()

---

## 5. What to Do With These Results

The engine found candidates — now you need to stress test them:

**Out-of-sample test:** Take the top strategy and run it on a *different* time period (e.g., train on 2010–2019, test on 2020–2025). Does it hold up?

**Cross-asset test:** Does it work on a different ticker in the same asset class? A momentum strategy that only works on NVDA but not MSFT or QQQ is probably overfit to NVDA's specific history.

**Deflated Sharpe Ratio (DSR):** Accounts for the number of strategies tried. If you tested 180 combinations, the expected maximum Sharpe under the null hypothesis (no edge) is much higher than 0. Bailey & Lopez de Prado's DSR corrects for this.

A rough rule of thumb: divide the backtest Sharpe by `sqrt(log(n_strategies_tested))` and see if it still looks good.

In [ ]:
import numpy as np

n_tested = len(results)
top_sharpe = results.iloc[0]['sharpe']
deflation_factor = np.sqrt(np.log(n_tested)) if n_tested > 1 else 1.0
deflated_sharpe = top_sharpe / deflation_factor

print(f'Strategies evaluated:  {n_tested}')
print(f'Top in-sample Sharpe:  {top_sharpe:.3f}')
print(f'Deflation factor:      {deflation_factor:.3f}  (sqrt(log({n_tested})))')
print(f'Rough deflated Sharpe: {deflated_sharpe:.3f}')
print()
print('A deflated Sharpe above ~0.5 is worth investigating further.')
print('Below that, the edge may be noise from searching many combinations.')